# 019 — 016 D + 007 H/A, complete H/A/D output

D: frozen public_model.pkl from 016, threshold 0.5, bridge LIVE gaps <=10 frames, remove D shorter than 25 frames. No refitting or threshold search. H/A: 007 cleaned confident anchors and nearest defensive-line rule; configuration frozen from 005/007. Recompute H/A with the NEW D barriers, never reuse the old D heuristic as final output.

007 selected by saved conditional H/A macroF1 on true H/A frames in the first four public minutes, before assessing the hybrid. Offline contexts and previously explored labels mean this is diagnostic selection, not an independent test.

X removal is an algorithmic rule, not a display change: extend first/last valid H/A anchor to the ends of each LIVE interval. If the interval has no confident anchors, use the closer observed cleaned team to the ball and temporally fill missing evidence inside that interval. If no such observation exists, use the closest-time available team evidence in the SAME half, possibly across a D boundary. This last fallback is explicitly logged as uncertain. No coordinate observations are fabricated. Outside the official half spans assign D (pre-match/halftime/post-match). Thus all 199319 frames have H/A/D, zero X.

All inference code in this notebook. Reuse preparation/functions from parent notebooks 007 and 016 without running their training/selection cells. Requires original local assets and feature preparation 008/broadcast alignment (sol/010_dead_logistic/prepare_broadcast_labels.ipynb); models 009/010/016 remain unchanged. Outputs and review video stay local_review. Approved libraries only. The public prefix was used to train 016; the last minute was excluded from its fit but has been repeatedly inspected. Full-match offline input context is used. Public-five-minute metrics mix training and diagnostics; full-match quality is unknown.


In [ ]:
from pathlib import Path
import json,sys,hashlib,pickle,time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'games.py').exists())
sys.path.insert(0,str(ROOT))
from games import GAME,GAMES,half_video_blocks,asset_paths
from evaluation import possession_metrics as pm
EXPERIMENT='019_boost016_clean007'
OUT=ROOT/'local_review/experiments'/EXPERIMENT;OUT.mkdir(parents=True,exist_ok=True)
N=199319;FPS=25;BLOCKS=half_video_blocks(GAME)
def progress(stage,**kw):(OUT/'progress.json').write_text(json.dumps(dict(stage=stage,**kw),indent=2))
# Rebuild probabilities from frozen 016 weights, without fitting or reading private labels.
# Восстанавливаем вероятности из неизменных весов 016, без обучения и закрытой разметки.
progress('016 inference')
parent016=json.loads((ROOT/'sol/016_dead_broadcast_features/experiment.ipynb').read_text())
denv={'EXPERIMENT':EXPERIMENT}
exec(''.join(next(c for c in parent016['cells'] if c.get('id')=='phase-0')['source']),denv)
with (ROOT/'sol/016_dead_broadcast_features/public_model.pkl').open('rb') as f:dmodel=pickle.load(f)
probability=denv['predict'](dmodel,np.arange(N));filtered=np.zeros(N,bool)
for _,a,b in BLOCKS:filtered[a:b+1]=denv['filter_segment'](probability[a:b+1]>=.5,25)
reference_path=ROOT/'local_review/experiments/016_dead_broadcast_features/public_p_D.npy'
if reference_path.exists():
 reference=np.load(reference_path);assert np.allclose(probability,reference,atol=1e-6)
np.save(OUT/'p_D.npy',probability);np.save(OUT/'dead_mask.npy',filtered)
del denv,dmodel
# Select only the existing H/A strategy, using the prefix; no new parameter search.
# Выбираем только существующую стратегию H/A по префиксу, без нового подбора параметров.
gt=pm.read_label_csv(ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv')
prefix=(gt.frame<17185)&gt.label.isin(['H','A']);comparison=[]
for name,file in [('001_box5_vote100','box5_vote100.csv'),('004_mass_near_ball','submission.csv'),('005_confident_anchors','submission.csv'),('007_clean_anchors','submission.csv')]:
 p=pm.read_label_csv(ROOT/'local_review/experiments'/name/file).label.to_numpy();y=gt.label.to_numpy();f1=[]
 for label in ['H','A']:
  actual=(y==label)&prefix.to_numpy();guessed=(p==label)&prefix.to_numpy()
  f1.append(2*np.count_nonzero(actual&guessed)/max(actual.sum()+guessed.sum(),1))
 comparison.append(dict(experiment=name,prefix_HA_macroF1=float(np.mean(f1))))
assert max(comparison,key=lambda row:row['prefix_HA_macroF1'])['experiment']=='007_clean_anchors'
(OUT/'ha_selection.json').write_text(json.dumps(comparison,indent=2))


In [ ]:
# Run preparation and algorithm definitions only, keeping all outputs in 019.
# Запускаем только подготовку и определения 007; все выходы остаются в 019.
progress('007 preparation')
parent007=json.loads((ROOT/'sol/007_clean_anchors/experiment.ipynb').read_text())
for index in [1,3,4,6,8]:
 source=''.join(parent007['cells'][index]['source'])
 if index==1:
  source=source.replace('EXPERIMENT = "007_clean_anchors"','EXPERIMENT = "019_boost016_clean007"')
  source=source.replace('ROOT / "sol" / EXPERIMENT / "grid.json"','ROOT / "sol/007_clean_anchors/grid.json"')
  source=source.replace('import matplotlib.pyplot as plt\n','')
 exec(compile(source,f'007 cell {index}','exec'))
# Replace legacy D completely before running the H/A solver.
# Полностью заменяем старый D перед запуском алгоритма H/A.
dead=filtered.copy()
selected=json.loads((ROOT/'sol/007_clean_anchors/selected_training_config.json').read_text())
progress('019 full inference')
labels,stats,gaps=predict_interval(selected,0,N,collect=True)
original_x=labels=='X';fallback_counts={};graded=np.zeros(N,bool)
def fill_nearest_time(values):
 valid=np.flatnonzero(np.isin(values,['H','A']))
 assert len(valid)>0
 missing=np.flatnonzero(~np.isin(values,['H','A']))
 if len(missing):
  right=np.searchsorted(valid,missing);lo=valid[np.maximum(right-1,0)];hi=valid[np.minimum(right,len(valid)-1)]
  chosen=np.where(missing-lo<=hi-missing,lo,hi)
  values[missing]=values[chosen]
 return values
# Fill only missing labels. Never change a confident anchor or a D decision.
# Заполняем только отсутствующие метки: уверенные якоря и решения D неизменны.
for half,first,last in BLOCKS:
 graded[first:last+1]=True
 nearest_evidence=np.full(last-first+1,'X',dtype='<U1')
 d=nearest[first:last+1];available=np.isfinite(d).any(axis=1)
 winner=np.argmin(d,axis=1)
 nearest_evidence[available]=np.array(['H','A'])[winner[available]]
 forced=outside_owner[first:last+1]!=3
 nearest_evidence[forced]=np.array(['H','A'])[outside_owner[first:last+1][forced]]
 assert np.isin(nearest_evidence,['H','A']).any(), 'No team evidence in entire half'
 half_prior=fill_nearest_time(nearest_evidence.copy())
 counts=dict(anchor_edge_fill=0,nearest_ball_fallback=0,same_half_fallback=0)
 states=dead[first:last+1]
 for a,b in runs_of(states):
  if states[a]:continue
  lo,hi=first+a,first+b;part=labels[lo:hi].copy();missing=part=='X'
  if not missing.any():continue
  if np.isin(part,['H','A']).any():
   counts['anchor_edge_fill']+=int(missing.sum());part=fill_nearest_time(part)
  else:
   evidence=nearest_evidence[a:b].copy()
   if np.isin(evidence,['H','A']).any():
    counts['nearest_ball_fallback']+=len(part);part=fill_nearest_time(evidence)
   else:
    counts['same_half_fallback']+=len(part);part=half_prior[a:b].copy()
  labels[lo:hi]=part
 fallback_counts[str(half)]=counts
labels[~graded]='D'
assert not np.any(labels=='X')
assert np.array_equal(labels[graded]=='D',filtered[graded])
assert np.array_equal(labels[graded&~original_x],predict_interval(selected,0,N)[0][graded&~original_x])
result=pd.DataFrame({'frame':np.arange(N),'label':labels});assert pm.validate_submission(result,N) is None
result.to_csv(OUT/'submission.csv',index=False)
# Exact start-of-half audit, including the second half pre-kickoff coverage.
# Точная проверка начал таймов, включая участок до второго розыгрыша.
audit={str(h):dict(first_frame=a,last_frame=b,first_label=str(labels[a]),X_frames=int((labels[a:b+1]=='X').sum()),initial_10s_X=int((labels[a:a+250]=='X').sum())) for h,a,b in BLOCKS}
(OUT/'coverage.json').write_text(json.dumps(dict(total_X=int((labels=='X').sum()),original_X_in_halves=int((original_x&graded).sum()),fallback_counts=fallback_counts,halves=audit,outside_half_policy='D'),indent=2))
print(json.dumps(audit,indent=2))


In [ ]:
# Metrics use only public labels, with full offline inference context.
# Метрики считаются только по публичным меткам с полным офлайн-контекстом.
def score_frames(frames):
 y=pm.label_codes(gt.set_index('frame').loc[frames,'label'].to_numpy())
 p=pm.label_codes(result.set_index('frame').loc[frames,'label'].to_numpy())
 runs=pm.valid_runs(y);cm=pm.scored_confusion(pm.confusion(y,p))
 out=dict(spellF1=pm.spell_scores(y,pm.fill_abstentions(p,runs),runs)['mean_f1'],macroF1=pm.macro_f1(cm),classes={})
 for i,label in enumerate(['H','A','D']):
  precision,recall,f1=pm.per_class_prf(cm,i)
  out['classes'][label]=dict(precision=precision,recall=recall,F1=f1,support=int(cm[i].sum()),predicted=int(cm[:,i].sum()))
 assert np.isclose(np.mean([v['F1'] for v in out['classes'].values()]),out['macroF1'])
 return out
public=gt.loc[gt.label.isin(['H','A','D']),'frame'].to_numpy()
metrics=dict(public_5min=score_frames(public),last_minute=score_frames(public[6000:]),
 context='016 trained on public prefix; H/A selected on public prefix; last minute repeatedly viewed diagnostic; full offline inputs, no refit')
(OUT/'metrics.json').write_text(json.dumps(metrics,indent=2))
config=dict(D_parent='016_dead_broadcast_features',D_model='public_model.pkl',threshold=.5,min_dead_frames=25,max_live_gap_frames=10,
 HA_parent='007_clean_anchors',HA_config=selected,no_X_policy='interval anchor extension, nearest-ball fallback, same-half temporal fallback; D outside halves',
 model_sha256=hashlib.sha256((ROOT/'sol/016_dead_broadcast_features/public_model.pkl').read_bytes()).hexdigest())
(OUT/'model_config.json').write_text(json.dumps(config,indent=2))
progress('completed',X_frames=0)
print(json.dumps(metrics,indent=2))
